# 08g Stable Center + Local Tangent Postprocess v1

08g는 08f의 safety-push 방식 대신, 더 local한 `center+tangent` 방식의 주행 후처리를 검토한다.

핵심 아이디어:

- 두 레인이 안정적이면 아래쪽 local y 구간에서 중앙선을 만들고, 그 중앙선의 lookahead x와 기울기로 조향한다.
- 두 레인이 보이지만 중앙선이 불안정하면 center를 믿지 않고 평균 tangent와 마지막 steer를 섞는다.
- 한쪽 레인만 보이면 left/right를 복원하지 않고, local tangent만 잠깐 사용한다.
- 레인을 잃으면 마지막 steer를 짧게 유지한 뒤 decay한다.

여기서 lookahead는 멀리 보는 값이 아니라, `near/mid/far`보다 조금 앞쪽에 있는 local 목표점이다.


## 안정성 판정

`both_stable` 판정은 center x 값의 표준편차를 그대로 보지 않는다. 코너에서는 정상적인 중앙선도 y에 따라 x가 변하기 때문이다.

따라서 세 개의 local center point가 하나의 짧은 직선으로 잘 설명되는지를 본다.

- gap sanity: 좌우 레인 간격이 너무 좁거나 넓지 않은가
- gap change: local 구간 안에서 gap이 과도하게 변하지 않는가
- center fit residual: 세 center point가 하나의 local 직선 위에 놓이는가
- center jump: 이전 안정 center와 비교해 lookahead x가 갑자기 튀지 않는가
- heading difference: 두 레인의 local tangent가 너무 다르지 않은가


In [ ]:

from __future__ import annotations

import csv
import json
from pathlib import Path

import cv2
import numpy as np


BASE = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild")
REVIEW_ROOT = BASE / "review_outputs" / "08g_stable_center_tangent_postprocess_v1"
VIDEO_DIR = REVIEW_ROOT / "videos"
TABLE_DIR = REVIEW_ROOT / "tables"
CONFIG_DIR = REVIEW_ROOT / "config"
FRAME_REVIEW_DIR = REVIEW_ROOT / "frame_review"
for d in [REVIEW_ROOT, VIDEO_DIR, TABLE_DIR, CONFIG_DIR, FRAME_REVIEW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PKG10 = BASE / "review_outputs" / "10_pi_runtime_latency_sequence_validation_v1" / "pkg"
RECORDS_CSV = PKG10 / "t" / "records_manifest.csv"
DECODED_JSONL = PKG10 / "r" / "ref_decoded.jsonl"

RAW_W = 1296
RAW_H = 972
IMAGE_CENTER_X = RAW_W / 2.0
HALF_W = RAW_W / 2.0

FIELD3_VIDEO_PATH = VIDEO_DIR / "field3_08g_stable_center_tangent_postprocess_v1.mp4"
FIELD3_SEQUENCE_CSV = TABLE_DIR / "field3_08g_stable_center_tangent_postprocess_v1.csv"
CONFIG_JSON = CONFIG_DIR / "lane_behavior_08g_stable_center_tangent_v1.json"
SUMMARY_JSON = REVIEW_ROOT / "stable_center_tangent_summary_v1.json"
SAMPLES_DIR = FRAME_REVIEW_DIR / "field3_samples"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)


LANE_BEHAVIOR_08G = {
    "name": "stable_center_tangent_v1",
    "description": "Local center line when two lanes are stable; local tangent only when one lane or unstable pair is visible.",

    # 1.0 is image bottom, 0.0 is image top. All points stay near the car.
    "near_y_ratio": 0.96,
    "mid_y_ratio": 0.90,
    "far_y_ratio": 0.84,
    "lookahead_y_ratio": 0.82,

    # Internal robustness gates. Keep these ratio-based so they are not tied to one resolution.
    "min_points": 4,
    "min_y_span_ratio": 0.055,
    "max_extrapolate_ratio": 0.16,
    "min_pair_gap_ratio": 0.18,
    "max_pair_gap_ratio": 1.12,
    "max_gap_change_ratio": 0.26,
    "center_fit_residual_ratio": 0.035,
    "max_center_jump_ratio": 0.22,
    "max_heading_diff": 3.50,

    # Field-tuning knobs.
    "center_gain": 0.75,
    "slope_gain": 0.38,
    "unstable_slope_gain": 0.34,
    "single_slope_gain": 0.46,
    "max_steer_norm": 0.78,

    # Memory only stabilizes short-term steering. It does not track lane identity.
    "steer_alpha": 0.72,
    "unstable_alpha": 0.35,
    "single_alpha": 0.62,
    "lost_hold_frames": 3,
    "lost_decay": 0.86,

    # Runtime/state-machine speed suggestions.
    "both_speed_scale": 1.00,
    "unstable_speed_scale": 0.72,
    "single_speed_scale": 0.58,
    "lost_speed_scale": 0.38,
}


def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def read_records_csv(path):
    with Path(path).open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def imread_bgr_unicode(path):
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img


def write_image_unicode(path, img):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), 92])
    if not ok:
        raise RuntimeError(path)
    buf.tofile(str(path))


In [ ]:

def normalize_lane_points(lane):
    pts = np.array(lane.get("points", []), dtype=np.float32)
    if pts.ndim != 2 or pts.shape[1] != 2:
        return np.zeros((0, 2), dtype=np.float32)
    valid = (
        np.isfinite(pts[:, 0])
        & np.isfinite(pts[:, 1])
        & (pts[:, 0] >= -RAW_W * 0.25)
        & (pts[:, 0] <= RAW_W * 1.25)
        & (pts[:, 1] >= 0)
        & (pts[:, 1] <= RAW_H)
    )
    return pts[valid]


def x_at_y(points, y_query):
    pts = np.asarray(points, dtype=np.float32)
    if len(pts) < 2:
        return None
    order = np.argsort(pts[:, 1])
    ys = pts[order, 1]
    xs = pts[order, 0]
    uniq_y, uniq_idx = np.unique(ys, return_index=True)
    ys = uniq_y
    xs = xs[uniq_idx]
    if len(ys) < 2:
        return None

    yq = float(y_query)
    if ys[0] <= yq <= ys[-1]:
        return float(np.interp(yq, ys, xs))
    if yq < ys[0]:
        y0, y1 = float(ys[0]), float(ys[1])
        x0, x1 = float(xs[0]), float(xs[1])
    else:
        y0, y1 = float(ys[-2]), float(ys[-1])
        x0, x1 = float(xs[-2]), float(xs[-1])
    if abs(y1 - y0) < 1e-6:
        return x0
    return float(x0 + (x1 - x0) * ((yq - y0) / (y1 - y0)))


def lane_feature(lane, cfg):
    pts = normalize_lane_points(lane)
    if len(pts) < int(cfg["min_points"]):
        return None

    y_span = float(pts[:, 1].max() - pts[:, 1].min())
    if y_span < float(cfg["min_y_span_ratio"]) * RAW_H:
        return None

    ys = {name: float(cfg[f"{name}_y_ratio"]) * RAW_H for name in ["near", "mid", "far"]}
    query_ys = np.array([ys["near"], ys["mid"], ys["far"]], dtype=np.float32)

    min_y = float(pts[:, 1].min())
    max_y = float(pts[:, 1].max())
    extrapolate_dist = 0.0
    for yq in query_ys:
        if yq < min_y:
            extrapolate_dist = max(extrapolate_dist, min_y - float(yq))
        elif yq > max_y:
            extrapolate_dist = max(extrapolate_dist, float(yq) - max_y)
    if extrapolate_dist > float(cfg["max_extrapolate_ratio"]) * RAW_H:
        return None

    x_near = x_at_y(pts, ys["near"])
    x_mid = x_at_y(pts, ys["mid"])
    x_far = x_at_y(pts, ys["far"])
    if x_near is None or x_mid is None or x_far is None:
        return None

    # Positive slope means x moves image-right as the sampled point goes farther forward.
    local_slope = (float(x_far) - float(x_near)) / max(1.0, ys["near"] - ys["far"])
    inside_count = int(((query_ys >= min_y) & (query_ys <= max_y)).sum())
    span_score = clamp(y_span / (RAW_H * 0.28), 0.0, 1.0)
    point_score = clamp(len(pts) / 22.0, 0.0, 1.0)
    conf_score = clamp(float(lane.get("conf", 0.0)) / 0.85, 0.0, 1.0)
    coverage_score = inside_count / 3.0
    reliability = 0.35 * conf_score + 0.25 * span_score + 0.20 * point_score + 0.20 * coverage_score

    return {
        "points": pts,
        "conf": float(lane.get("conf", 0.0)),
        "x_near": float(x_near),
        "x_mid": float(x_mid),
        "x_far": float(x_far),
        "slope": float(local_slope),
        "reliability": float(clamp(reliability, 0.0, 1.0)),
        "center_dist_ratio": float((x_mid - IMAGE_CENTER_X) / HALF_W),
        "y_span": y_span,
    }


def extract_lane_features(lanes, cfg):
    features = []
    for lane in lanes:
        feat = lane_feature(lane, cfg)
        if feat is not None:
            features.append(feat)
    features.sort(key=lambda f: (-f["reliability"], abs(f["center_dist_ratio"])))
    return features


def center_line_fit(centers):
    # Fit x = a*y + b over local center points, then measure x residual.
    ys = np.array([c[1] for c in centers], dtype=np.float32)
    xs = np.array([c[0] for c in centers], dtype=np.float32)
    a, b = np.polyfit(ys, xs, 1)
    pred = a * ys + b
    residual = float(np.sqrt(np.mean((xs - pred) ** 2)))
    return float(a), float(b), residual


def pair_metrics(a, b, memory, cfg):
    left, right = sorted([a, b], key=lambda f: f["x_mid"])
    gap_near = right["x_near"] - left["x_near"]
    gap_mid = right["x_mid"] - left["x_mid"]
    gap_far = right["x_far"] - left["x_far"]
    gaps = np.array([gap_near, gap_mid, gap_far], dtype=np.float32)
    min_gap = float(np.min(gaps))
    max_gap = float(np.max(gaps))
    gap_change_ratio = float((max_gap - min_gap) / RAW_W)

    centers = [
        (0.5 * (left["x_near"] + right["x_near"]), float(cfg["near_y_ratio"]) * RAW_H),
        (0.5 * (left["x_mid"] + right["x_mid"]), float(cfg["mid_y_ratio"]) * RAW_H),
        (0.5 * (left["x_far"] + right["x_far"]), float(cfg["far_y_ratio"]) * RAW_H),
    ]
    fit_a, fit_b, fit_residual_px = center_line_fit(centers)
    lookahead_y = float(cfg["lookahead_y_ratio"]) * RAW_H
    lookahead_x = float(fit_a * lookahead_y + fit_b)
    center_error = (lookahead_x - IMAGE_CENTER_X) / HALF_W

    # Convert x=a*y+b to the same forward convention as lane_feature slope.
    center_slope = -fit_a
    avg_lane_slope = 0.5 * (left["slope"] + right["slope"])
    heading_diff = abs(left["slope"] - right["slope"])

    jump_ratio = 0.0
    if memory.get("has_center", False):
        jump_ratio = abs(lookahead_x - float(memory["last_center_x"])) / RAW_W

    reasons = []
    if min_gap < float(cfg["min_pair_gap_ratio"]) * RAW_W:
        reasons.append("gap_too_small")
    if max_gap > float(cfg["max_pair_gap_ratio"]) * RAW_W:
        reasons.append("gap_too_large")
    if gap_change_ratio > float(cfg["max_gap_change_ratio"]):
        reasons.append("gap_changes")
    if fit_residual_px / RAW_W > float(cfg["center_fit_residual_ratio"]):
        reasons.append("center_not_line")
    if jump_ratio > float(cfg["max_center_jump_ratio"]):
        reasons.append("center_jump")
    if heading_diff > float(cfg["max_heading_diff"]):
        reasons.append("heading_diff")

    return {
        "left": left,
        "right": right,
        "centers": centers,
        "lookahead_x": float(lookahead_x),
        "lookahead_y": float(lookahead_y),
        "center_error": float(center_error),
        "center_slope": float(center_slope),
        "avg_lane_slope": float(avg_lane_slope),
        "gap_mid": float(gap_mid),
        "gap_change_ratio": float(gap_change_ratio),
        "fit_residual_ratio": float(fit_residual_px / RAW_W),
        "jump_ratio": float(jump_ratio),
        "heading_diff": float(heading_diff),
        "stable": len(reasons) == 0,
        "unstable_reasons": reasons,
    }


def choose_best_pair(features, memory, cfg):
    if len(features) < 2:
        return None
    pairs = []
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            m = pair_metrics(features[i], features[j], memory, cfg)
            reliability = 0.5 * (features[i]["reliability"] + features[j]["reliability"])
            center_near = 1.0 - min(1.0, abs(m["center_error"]))
            gap_ok = 1.0 - min(1.0, m["gap_change_ratio"] / max(1e-6, float(cfg["max_gap_change_ratio"])))
            jump_ok = 1.0 - min(1.0, m["jump_ratio"] / max(1e-6, float(cfg["max_center_jump_ratio"])))
            score = (10.0 if m["stable"] else 0.0) + 2.0 * reliability + center_near + 0.5 * gap_ok + 0.5 * jump_ok
            pairs.append((score, m))
    pairs.sort(key=lambda item: item[0], reverse=True)
    return pairs[0][1]


In [ ]:

def init_drive_memory_08g():
    return {
        "last_steer": 0.0,
        "last_slope": 0.0,
        "last_center_x": IMAGE_CENTER_X,
        "has_center": False,
        "lost_frames": 0,
    }


def update_drive_08g(lanes, memory, cfg):
    features = extract_lane_features(lanes, cfg)
    raw_steer = float(memory.get("last_steer", 0.0))
    lane_state = "lost"
    stable_forward = False
    speed_scale = float(cfg["lost_speed_scale"])
    debug = {"feature_count": len(features), "visible_lane_count": len(lanes)}

    if len(features) >= 2:
        pair = choose_best_pair(features, memory, cfg)
        if pair is not None and pair["stable"]:
            raw_steer = float(cfg["center_gain"]) * pair["center_error"] + float(cfg["slope_gain"]) * pair["center_slope"]
            lane_state = "both_stable"
            stable_forward = abs(pair["center_error"]) <= 0.30 and abs(pair["center_slope"]) <= 1.35
            speed_scale = float(cfg["both_speed_scale"])
            memory["last_center_x"] = pair["lookahead_x"]
            memory["has_center"] = True
            memory["last_slope"] = pair["center_slope"]
            debug.update(pair)
        elif pair is not None:
            slope = pair["avg_lane_slope"]
            slope_steer = float(cfg["unstable_slope_gain"]) * slope
            raw_steer = (1.0 - float(cfg["unstable_alpha"])) * float(memory["last_steer"]) + float(cfg["unstable_alpha"]) * slope_steer
            lane_state = "both_unstable"
            speed_scale = float(cfg["unstable_speed_scale"])
            memory["last_slope"] = 0.70 * float(memory.get("last_slope", 0.0)) + 0.30 * slope
            debug.update(pair)
    elif len(features) == 1:
        slope = features[0]["slope"]
        slope_steer = float(cfg["single_slope_gain"]) * slope
        raw_steer = (1.0 - float(cfg["single_alpha"])) * float(memory["last_steer"]) + float(cfg["single_alpha"]) * slope_steer
        lane_state = "single_slope"
        speed_scale = float(cfg["single_speed_scale"])
        memory["last_slope"] = 0.65 * float(memory.get("last_slope", 0.0)) + 0.35 * slope
        debug.update({"single_slope": float(slope), "single_x_mid": features[0]["x_mid"], "single_conf": features[0]["conf"]})

    if features:
        memory["lost_frames"] = 0
        raw_steer = clamp(raw_steer, -float(cfg["max_steer_norm"]), float(cfg["max_steer_norm"]))
        steer = (1.0 - float(cfg["steer_alpha"])) * float(memory["last_steer"]) + float(cfg["steer_alpha"]) * raw_steer
    else:
        memory["lost_frames"] = int(memory.get("lost_frames", 0)) + 1
        if memory["lost_frames"] <= int(cfg["lost_hold_frames"]):
            steer = float(memory["last_steer"])
            lane_state = "lost_hold"
        else:
            steer = float(memory["last_steer"]) * float(cfg["lost_decay"])
            lane_state = "lost_decay"
        raw_steer = steer
        speed_scale = float(cfg["lost_speed_scale"])

    steer = clamp(steer, -float(cfg["max_steer_norm"]), float(cfg["max_steer_norm"]))
    memory["last_steer"] = steer

    return {
        "steer_norm": float(steer),
        "raw_steer": float(raw_steer),
        "speed_scale": float(speed_scale),
        "lane_state": lane_state,
        "stable_forward": bool(stable_forward),
        # Compatibility with the current runtime/state debug vocabulary.
        "mode": lane_state,
        "speed_hint": float(speed_scale),
        "departure_risk": {"both_stable": 0.0, "both_unstable": 0.35, "single_slope": 0.45, "lost_hold": 0.70, "lost_decay": 0.85}.get(lane_state, 0.5),
        "lane_confidence": {"both_stable": 0.90, "both_unstable": 0.55, "single_slope": 0.45, "lost_hold": 0.10, "lost_decay": 0.05}.get(lane_state, 0.30),
        "feature_count": int(len(features)),
        "visible_lane_count": int(len(lanes)),
        "features": features,
        "debug": debug,
    }


## Field3 replay

아래 셀은 field3 sequence 전체에 대해 08g를 replay하고 overlay video / CSV / sample frame을 저장한다.


In [ ]:

def load_field3_sequence():
    records = read_records_csv(RECORDS_CSV)
    decoded = read_jsonl(DECODED_JSONL)
    decoded_by_key = {row["key"]: row for row in decoded}
    seq = [r for r in records if r["set"] == "field3" and r["role"] == "sequence"]
    seq.sort(key=lambda r: int(r["order"]))
    return seq, decoded_by_key


def draw_text(img, lines, x=14, y=28, dy=24):
    for idx, text in enumerate(lines):
        yy = y + idx * dy
        cv2.putText(img, text, (x, yy), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(img, text, (x, yy), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 1, cv2.LINE_AA)


def draw_lane_polyline(img, lane, color, thickness=3):
    pts = normalize_lane_points(lane).astype(np.int32)
    if len(pts) >= 2:
        cv2.polylines(img, [pts.reshape(-1, 1, 2)], False, color, thickness, cv2.LINE_AA)


def draw_center_debug(img, result, cfg):
    d = result.get("debug", {})
    near_y = int(float(cfg["near_y_ratio"]) * RAW_H)
    mid_y = int(float(cfg["mid_y_ratio"]) * RAW_H)
    far_y = int(float(cfg["far_y_ratio"]) * RAW_H)
    look_y = int(float(cfg["lookahead_y_ratio"]) * RAW_H)
    for y, c in [(near_y, (70, 70, 70)), (mid_y, (90, 90, 90)), (far_y, (110, 110, 110)), (look_y, (0, 200, 255))]:
        cv2.line(img, (0, y), (RAW_W - 1, y), c, 1, cv2.LINE_AA)

    if "centers" in d:
        centers = [(int(x), int(y)) for x, y in d["centers"]]
        for p in centers:
            cv2.circle(img, p, 7, (0, 255, 255), -1, cv2.LINE_AA)
        if len(centers) >= 2:
            cv2.line(img, centers[0], centers[-1], (0, 255, 255), 3, cv2.LINE_AA)
        lx = int(d.get("lookahead_x", IMAGE_CENTER_X))
        ly = int(d.get("lookahead_y", look_y))
        cv2.circle(img, (lx, ly), 10, (0, 128, 255), -1, cv2.LINE_AA)
        cv2.line(img, (int(IMAGE_CENTER_X), RAW_H - 1), (lx, ly), (0, 128, 255), 2, cv2.LINE_AA)
    elif "single_x_mid" in d:
        for feat in result.get("features", []):
            p1 = (int(feat["x_near"]), near_y)
            p2 = (int(feat["x_far"]), far_y)
            cv2.line(img, p1, p2, (255, 255, 0), 3, cv2.LINE_AA)
            cv2.circle(img, p1, 7, (255, 255, 0), -1, cv2.LINE_AA)
            cv2.circle(img, p2, 7, (255, 255, 0), -1, cv2.LINE_AA)

    cv2.line(img, (int(IMAGE_CENTER_X), 0), (int(IMAGE_CENTER_X), RAW_H - 1), (200, 200, 200), 2, cv2.LINE_AA)


def overlay_drive(rec, result, cfg, decoded_by_key, scale_width=960):
    img = imread_bgr_unicode(Path(rec["source_path_local"]))
    row = decoded_by_key[rec["key"]]
    lanes = row.get("lanes", [])
    colors = [(80, 220, 255), (255, 160, 80), (180, 120, 255), (120, 255, 120)]
    for idx, lane in enumerate(lanes):
        draw_lane_polyline(img, lane, colors[idx % len(colors)], thickness=3)
    draw_center_debug(img, result, cfg)

    d = result.get("debug", {})
    reasons = ",".join(d.get("unstable_reasons", [])[:3]) if d.get("unstable_reasons") else "-"
    slope = d.get("center_slope", d.get("single_slope", 0.0))
    lines = [
        f'{int(rec["order"]):04d} {result["lane_state"]} steer={result["steer_norm"]:+.3f} speed={result["speed_scale"]:.2f}',
        f'features={result["feature_count"]} stable={int(result["stable_forward"])} center_err={d.get("center_error", 0.0):+.3f} slope={slope:+.3f}',
        f'resid={d.get("fit_residual_ratio", 0.0):.3f} jump={d.get("jump_ratio", 0.0):.3f} gapchg={d.get("gap_change_ratio", 0.0):.3f} reason={reasons}',
    ]
    draw_text(img, lines)
    if scale_width and img.shape[1] != scale_width:
        scale = scale_width / img.shape[1]
        img = cv2.resize(img, (scale_width, int(img.shape[0] * scale)), interpolation=cv2.INTER_AREA)
    return img


def write_video(frames, out_path, fps=12):
    if not frames:
        raise RuntimeError("no frames")
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    if not writer.isOpened():
        raise RuntimeError(f"cannot open video writer: {out_path}")
    for frame in frames:
        writer.write(frame)
    writer.release()


def run_field3_replay():
    seq, decoded_by_key = load_field3_sequence()
    memory = init_drive_memory_08g()
    rows = []
    frames = []
    mode_counts = {}
    sample_orders = {0, 30, 60, 90, 120, 150, 180, 220, 260, 300, 340}

    for rec in seq:
        lanes = decoded_by_key[rec["key"]].get("lanes", [])
        result = update_drive_08g(lanes, memory, LANE_BEHAVIOR_08G)
        mode_counts[result["lane_state"]] = mode_counts.get(result["lane_state"], 0) + 1
        d = result.get("debug", {})
        rows.append({
            "order": int(rec["order"]),
            "key": rec["key"],
            "lane_state": result["lane_state"],
            "steer_norm": result["steer_norm"],
            "raw_steer": result["raw_steer"],
            "speed_scale": result["speed_scale"],
            "stable_forward": int(result["stable_forward"]),
            "feature_count": result["feature_count"],
            "visible_lane_count": result["visible_lane_count"],
            "center_error": d.get("center_error"),
            "center_slope": d.get("center_slope"),
            "single_slope": d.get("single_slope"),
            "fit_residual_ratio": d.get("fit_residual_ratio"),
            "jump_ratio": d.get("jump_ratio"),
            "gap_change_ratio": d.get("gap_change_ratio"),
            "heading_diff": d.get("heading_diff"),
            "unstable_reasons": ";".join(d.get("unstable_reasons", [])) if d.get("unstable_reasons") else "",
        })
        frame = overlay_drive(rec, result, LANE_BEHAVIOR_08G, decoded_by_key, scale_width=960)
        frames.append(frame)
        if int(rec["order"]) in sample_orders:
            write_image_unicode(SAMPLES_DIR / f"field3_{int(rec['order']):04d}_{result['lane_state']}.jpg", frame)

    write_video(frames, FIELD3_VIDEO_PATH, fps=12)
    with FIELD3_SEQUENCE_CSV.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    write_json(CONFIG_JSON, LANE_BEHAVIOR_08G)

    summary = {
        "video": str(FIELD3_VIDEO_PATH),
        "table": str(FIELD3_SEQUENCE_CSV),
        "config": str(CONFIG_JSON),
        "samples_dir": str(SAMPLES_DIR),
        "frame_count": len(rows),
        "mode_counts": mode_counts,
        "mean_abs_steer": float(np.mean([abs(float(r["steer_norm"])) for r in rows])),
        "max_abs_steer": float(np.max([abs(float(r["steer_norm"])) for r in rows])),
        "mean_speed_scale": float(np.mean([float(r["speed_scale"]) for r in rows])),
        "min_speed_scale": float(np.min([float(r["speed_scale"]) for r in rows])),
    }
    write_json(SUMMARY_JSON, summary)
    return rows, summary


rows, summary = run_field3_replay()
print(json.dumps(summary, indent=2, ensure_ascii=False))
